In [50]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('GK_Questions.pdf')
pages = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
spilitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 30)
chunk_fun = spilitter.split_documents(pages)
chunk = [i.page_content for i in chunk_fun]
metadata = [i.metadata for i in chunk_fun]
metadata[0]

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-07-09T11:07:34+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-07-09T11:07:34+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'GK_Questions.pdf',
 'total_pages': 19,
 'page': 0,
 'page_label': '1'}

In [51]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding = SentenceTransformerEmbeddingFunction()

client = chromadb.PersistentClient(path='./Collection')
collection = client.get_or_create_collection(name='collection', embedding_function=embedding)

if collection.count()== 0:
    collection.add(
        documents=chunk,
        ids=[str(i) for i in range(len(chunk))],
        metadatas=metadata
    )
collection.count()   

37

In [ ]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv 
load_dotenv()
key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model='openai/gpt-oss-120b')

import ast
from langchain_core.tools import tool
import ast
from langchain_core.tools import tool

@tool
def calculator(expression: str):
    """This tool is useful for arithmetic operations."""
    try:
        response = ast.literal_eval(expression)
        return str(response)
    except Exception as e:
        return str(e)
    

In [ ]:
@tool
def retrive(query:str)->str:
    """do every users asking find in local document"""
    prompt = f"""do the query like symentic search {query}"""
    query_rewrites = llm.invoke(prompt)
    query_rewrites = query_rewrites.content()
    
    result = collection.query(query_texts=[query_rewrites] , n_results=5)
    document = result['documents'][0]
    distances = result['distances'][0]
    thresold = 1.0 
    print(distances)
    good_chunk = []
    for doc, dis in zip(document,distances):
        if dis < thresold:
            good_chunk.append(doc)
    if not good_chunk:
        return 'NOT RELEVENT CONTENT'    
    return '\n\n'.join(good_chunk)


In [73]:
tools_atribute= [calculator , retrive]
tool_name = {t.name : t for t in tools_atribute}
tool_name

{'calculator': StructuredTool(name='calculator', description='This tool is useful for arithmetic operations.', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x000001CFBD191E40>),
 'retrive': StructuredTool(name='retrive', description='do every users asking find in local document', args_schema=<class 'langchain_core.utils.pydantic.retrive'>, func=<function retrive at 0x000001CFBD191D00>)}

In [74]:
calculator_agent = llm.bind_tools([calculator])
retrive_agent = llm.bind_tools([retrive])

In [75]:
agents  = {
    "calculator_agent" : calculator_agent , 
    "retrive_agent" : retrive_agent
}

In [76]:
def find_agent(query):
    prompt = f"""
    Choose one tool for one work.

    If there are arithmetic operations, use calculator_agent.
    Every other question should use retrive_agent.

    Question: {query}

    Return ONLY one of:
    calculator_agent
    retrive_agent
    """

    response = llm.invoke(prompt)

    label = response.content.lower().strip()

    if label == "calculator_agent":
        return "calculator_agent"

    elif label == "retrive_agent":
        return "retrive_agent"

    else:
        return "general"

In [77]:
def search(query:str):
    label = find_agent(query=query)

    if label == "general":
        response = llm.invoke(label)
        return response.content 
    agent_search = agents[label]
    response = agent_search.invoke(query)

    if not response.tool_calls:
        return response.content 
    for call in response.tool_calls:
        args = call['args']
        name = call['name']

        result = tool_name[name].invoke(args)
    final_result = llm.invoke(result)
    return final_result.content

In [78]:
q1 = "how is happen 4*21?"
response = search(query=q1)

print(response)

Multiplying 4 by 21 is straightforward:

\[
4 \times 21 = (4 \times 20) + (4 \times 1)
\]

1. **4 × 20 = 80** (because 4 × 2 = 8, then add a zero)
2. **4 × 1 = 4**

Add the two partial results:

\[
80 + 4 = 84
\]

So, **4 × 21 = 84**.
